# Data Source: STM Metro Locations

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))

from stayrank.config import RAW_STM_STOPS_PATH, PROCESSED_DIR

## Load Data

In [2]:
stops = pd.read_csv(RAW_STM_STOPS_PATH)
stops.shape

(9188, 9)

In [3]:
stops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9188 entries, 0 to 9187
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   stop_id              9188 non-null   object 
 1   stop_code            9188 non-null   int64  
 2   stop_name            9188 non-null   object 
 3   stop_lat             9188 non-null   float64
 4   stop_lon             9188 non-null   float64
 5   stop_url             8986 non-null   object 
 6   location_type        9188 non-null   int64  
 7   parent_station       206 non-null    object 
 8   wheelchair_boarding  9188 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 646.2+ KB


In [4]:
stops.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding
0,STATION_M118,10118,STATION ANGRIGNON,45.446397,-73.603293,NaN,1,NaN,1
1,43,10118,Station Angrignon,45.446466,-73.603118,https://www.stm.info/fr/infos/reseaux/metro/an...,0,STATION_M118,1
2,43-01,10118,Station Angrignon,45.446319,-73.603835,NaN,2,STATION_M118,1
3,STATION_M120,10120,STATION MONK,45.451166,-73.593265,NaN,1,NaN,2
4,42,10120,Station Monk,45.451158,-73.593242,https://www.stm.info/fr/infos/reseaux/metro/monk,0,STATION_M120,2


## Inspect stop types

In [5]:
stops["location_type"].value_counts()

location_type
0    8986
2     134
1      68
Name: count, dtype: int64

In [12]:
stops[
    (stops["stop_name"].str.lower().str.startswith("station")) 
    & (stops.parent_station.isna())
].location_type.value_counts()

location_type
0    382
1     68
Name: count, dtype: int64

## Extract metro stations

In [16]:
KEEP_COLUMNS = ["stop_name", "stop_lat", "stop_lon"]

metro_stations = (
    stops.query("location_type == 1")
    [KEEP_COLUMNS]
    .copy()
)

metro_stations["stop_name"] = metro_stations["stop_name"].str.title()

metro_stations.head(20)

,stop_name,stop_lat,stop_lon
0,Station Angrignon,45.446397,-73.603293
3,Station Monk,45.451166,-73.593265
7,Station Jolicoeur,45.456783,-73.581991
10,Station Verdun,45.459432,-73.571654
14,Station De L'Église,45.461911,-73.567107
18,Station Lasalle,45.470631,-73.566281
21,Station Charlevoix,45.478467,-73.569366
24,Station Lionel-Groulx,45.482818,-73.579721
27,Station Atwater,45.490038,-73.585731
30,Station Guy-Concordia,45.495692,-73.579250


In [14]:
metro_stations.isna().sum()

stop_name    0
stop_lat     0
stop_lon     0
dtype: int64

## Save processed data

In [17]:
CLEAN_METRO_PATH = PROCESSED_DIR / "montreal_metro_stations.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

metro_stations.to_parquet(CLEAN_METRO_PATH, index=False)

CLEAN_METRO_PATH

WindowsPath('C:/Users/ngoum/Documents/coding/data/agentic-pyrank/data/processed/metro.parquet')